# Phase 3.1b - Level 4 Dynamic Policy Tuning

This notebook tunes the Phase 3.1 amount-bin dynamic decision policy. It does not retrain the LightGBM/XGBoost risk model and does not implement Level 5.

Goal: reduce over-blocking in Cost-A/B while preserving the Cost-C gain.

## 0. Run Contract

Run this notebook after:

1. `02_baselines_cost_metrics.ipynb`
2. `04_dynamic_cost_policy.ipynb`

Required inputs:

- `results/baseline_scores_{RUN_MODE}.csv`
- `results/dynamic_policy_metrics_{RUN_MODE}.csv`
- `results/global_cost_thresholds_{RUN_MODE}.csv`
- `results/amount_bin_thresholds_{RUN_MODE}.csv`

Leakage rule: all candidate choices are selected on validation only. Test is report-only after selection.

Math guard: under the original cost model, `C_FN = beta * TransactionAmt` and `C_FP = alpha * TransactionAmt`, the pure closed-form threshold is `alpha / (alpha + beta)` because `TransactionAmt` cancels. Therefore, this notebook does not claim that the closed-form threshold is amount-dynamic.

In [1]:
from pathlib import Path
import json
import os
import re
import sys
import warnings

import numpy as np
import pandas as pd

import matplotlib
if "COLAB_RELEASE_TAG" not in os.environ:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

SEED = 42
np.random.seed(SEED)

RUN_MODE = "sample_100k"  # "smoke", "sample_100k", "sample_200k", "sample_300k", or "full"
RUN_MODE_SAMPLE_ROWS = {
    "smoke": 10_000,
    "sample_100k": 100_000,
    "sample_200k": 200_000,
    "sample_300k": 300_000,
    "full": None,
}
if RUN_MODE not in RUN_MODE_SAMPLE_ROWS:
    raise ValueError(f"Unsupported RUN_MODE={RUN_MODE!r}")

RUN_OUTPUT_TAG = RUN_MODE
SAMPLE_ROWS_LABEL = RUN_MODE_SAMPLE_ROWS[RUN_MODE] if RUN_MODE_SAMPLE_ROWS[RUN_MODE] is not None else "full"

SELECTED_RISK_MODEL = "lightgbm_balanced"
MODEL_FALLBACK_ORDER = [
    "lightgbm_balanced",
    "xgboost_magic_style",
    "lightgbm_magic_style",
    "xgboost_scale_pos_weight",
    "random_forest_balanced",
    "logistic_regression_balanced",
]

COST_CONFIGS = {
    "Cost-A": {"alpha": 0.05, "beta": 1.0},
    "Cost-B": {"alpha": 0.10, "beta": 2.0},
    "Cost-C": {"alpha": 0.20, "beta": 5.0},
}

THRESHOLD_GRID = np.unique(np.concatenate([
    np.linspace(0.0, 1.0, 201),
    np.array([0.001, 0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30, 0.40, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]),
]))
LAMBDA_GRID = [0.00, 0.25, 0.50, 0.75, 1.00]
PRECISION_FLOOR_GRID = [0.05, 0.10, 0.15, 0.20, 0.25]
FP_COST_CAP_MULTIPLIER_GRID = [1.00, 1.25, 1.50, 2.00]
MIN_BIN_VALIDATION_ROWS = 100
MIN_BIN_VALIDATION_FRAUD = 5
GUARDED_SELECTOR_TOLERANCE = 1.05

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
MOUNT_GOOGLE_DRIVE = IN_COLAB
DEFAULT_COLAB_PROJECT_ROOT = (
    "/content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive"
    if IN_COLAB
    else ""
)
PROJECT_ROOT_OVERRIDE = os.environ.get("PROJECT_ROOT_OVERRIDE", DEFAULT_COLAB_PROJECT_ROOT).strip()

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Warning: Google Drive mount failed: {exc}")


def looks_like_project_root(path):
    path = Path(path)
    return (path / "notebooks").exists() and (
        (path / "notebooks" / "04b_dynamic_policy_tuning.ipynb").exists()
        or (path / "notebooks" / "04_dynamic_cost_policy.ipynb").exists()
        or (path / "requirements.txt").exists()
    )


def find_project_root(start=None):
    candidates = []
    if PROJECT_ROOT_OVERRIDE:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE))
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root))
    start = Path(start or Path.cwd()).resolve()
    candidates.extend([start, *start.parents])
    if IN_COLAB:
        candidates.extend([
            Path("/content/LLM-Assisted_Cost-Sensitive"),
            Path("/content/drive/MyDrive/LLM-Assisted_Cost-Sensitive"),
            Path("/content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive"),
            Path("/content/drive/MyDrive/HCMUTE_TMDT/HKII_Nam_3/Bao_Mat_TMDT/LLM-Assisted_Cost-Sensitive"),
        ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if not candidate.exists():
            continue
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if looks_like_project_root(resolved):
            return resolved
    return start


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "baseline_scores": RESULTS_DIR / f"baseline_scores_{RUN_OUTPUT_TAG}.csv",
    "dynamic_policy_metrics": RESULTS_DIR / f"dynamic_policy_metrics_{RUN_OUTPUT_TAG}.csv",
    "global_cost_thresholds": RESULTS_DIR / f"global_cost_thresholds_{RUN_OUTPUT_TAG}.csv",
    "amount_bin_thresholds": RESULTS_DIR / f"amount_bin_thresholds_{RUN_OUTPUT_TAG}.csv",
    "five_level_comparison": RESULTS_DIR / f"five_level_comparison_{RUN_OUTPUT_TAG}.csv",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"RUN_MODE={RUN_MODE} | RUN_OUTPUT_TAG={RUN_OUTPUT_TAG}")
for name, path in PATHS.items():
    print(f"{name}: {path} | exists={path.exists()}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive
RUN_MODE=sample_100k | RUN_OUTPUT_TAG=sample_100k
baseline_scores: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/baseline_scores_sample_100k.csv | exists=True
dynamic_policy_metrics: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/dynamic_policy_metrics_sample_100k.csv | exists=True
global_cost_thresholds: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/global_cost_thresholds_sample_100k.csv | exists=True
amount_bin_thresholds: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/amount_bin_thresholds_sample_100k.csv | exists=True
five_level_comparison: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/five_level_comparison_sample_100k.csv | exists=True


## 1. Load Phase 2 / Phase 3.1 Artifacts

In [2]:
REQUIRED_SCORE_COLUMNS = {
    "run_mode", "sample_rows", "split", "TransactionID", "isFraud",
    "TransactionAmt", "model", "score"
}
REQUIRED_INPUTS = [
    "baseline_scores",
    "dynamic_policy_metrics",
    "global_cost_thresholds",
    "amount_bin_thresholds",
]

missing_inputs = [name for name in REQUIRED_INPUTS if not PATHS[name].exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing required Phase 3.1b input files:\n"
        + "\n".join(f"- {name}: {PATHS[name]}" for name in missing_inputs)
        + "\nRun notebook 02 and notebook 04 first with the same RUN_MODE."
    )

scores_df = pd.read_csv(PATHS["baseline_scores"])
phase31_metrics_df = pd.read_csv(PATHS["dynamic_policy_metrics"])
global_thresholds_df = pd.read_csv(PATHS["global_cost_thresholds"])
amount_thresholds_df = pd.read_csv(PATHS["amount_bin_thresholds"])
five_level_df = pd.read_csv(PATHS["five_level_comparison"]) if PATHS["five_level_comparison"].exists() else pd.DataFrame()

missing_columns = sorted(REQUIRED_SCORE_COLUMNS - set(scores_df.columns))
if missing_columns:
    raise ValueError(f"baseline_scores is missing columns: {missing_columns}")

for name, df in [
    ("scores_df", scores_df),
    ("phase31_metrics_df", phase31_metrics_df),
    ("global_thresholds_df", global_thresholds_df),
    ("amount_thresholds_df", amount_thresholds_df),
]:
    if "run_mode" in df.columns:
        found_modes = set(df["run_mode"].astype(str))
        if not found_modes.issubset({RUN_MODE}):
            raise ValueError(f"{name} RUN_MODE mismatch. Expected {RUN_MODE}, found {sorted(found_modes)}")

scores_df["split"] = scores_df["split"].astype(str)
scores_df["model"] = scores_df["model"].astype(str)
scores_df["isFraud"] = scores_df["isFraud"].astype(int)
scores_df["TransactionAmt"] = scores_df["TransactionAmt"].fillna(0).astype(float)
scores_df["score"] = scores_df["score"].astype(float)

available_models = sorted(scores_df["model"].unique())
if SELECTED_RISK_MODEL not in available_models:
    fallback = next((name for name in MODEL_FALLBACK_ORDER if name in available_models), None)
    if fallback is None:
        raise ValueError(f"No supported risk model found. Available: {available_models}")
    print(f"Selected model {SELECTED_RISK_MODEL!r} not found. Falling back to {fallback!r}.")
    SELECTED_RISK_MODEL = fallback

policy_scores = scores_df[scores_df["model"] == SELECTED_RISK_MODEL].copy()
if not {"validation", "test"}.issubset(set(policy_scores["split"])):
    raise ValueError(f"Selected model {SELECTED_RISK_MODEL!r} must have validation and test splits.")

validation_df = policy_scores[policy_scores["split"] == "validation"].copy().reset_index(drop=True)
test_df = policy_scores[policy_scores["split"] == "test"].copy().reset_index(drop=True)

print(f"Selected risk model: {SELECTED_RISK_MODEL}")
display(policy_scores.groupby(["split", "model"]).agg(
    rows=("TransactionID", "count"),
    fraud_rate=("isFraud", "mean"),
    amount_mean=("TransactionAmt", "mean"),
    score_mean=("score", "mean"),
).reset_index())

print("Phase 3.1 test reference")
display(phase31_metrics_df[phase31_metrics_df["split"].eq("test")][[
    "level", "policy", "cost_config", "recall_fraud", "precision_fraud",
    "fp_cost", "total_cost", "cost_saving_vs_approve_all"
]].sort_values(["cost_config", "level"]))

Selected risk model: lightgbm_balanced


,split,model,rows,fraud_rate,amount_mean,score_mean
0,test,lightgbm_balanced,15000,0.020333,138.315390,0.213269
1,validation,lightgbm_balanced,15000,0.025133,138.821157,0.198650


Phase 3.1 test reference


,level,policy,cost_config,recall_fraud,precision_fraud,fp_cost,total_cost,cost_saving_vs_approve_all
1,3,level3_global_cost_threshold,Cost-A,0.567213,0.335271,2159.069448,20198.387418,17244.647541
7,4,level4_amount_bin_dynamic_threshold,Cost-A,0.655738,0.148368,4402.340947,21028.004927,16415.030032
3,3,level3_global_cost_threshold,Cost-B,0.567213,0.335271,4318.138897,40396.774837,34489.295082
9,4,level4_amount_bin_dynamic_threshold,Cost-B,0.655738,0.148368,8804.681893,42056.009854,32830.060064
5,3,level3_global_cost_threshold,Cost-C,0.731148,0.098716,57246.004751,109064.429634,78150.745163
11,4,level4_amount_bin_dynamic_threshold,Cost-C,0.714754,0.111623,38720.450946,102499.135839,84716.038958


## 2. Shared Metrics and Threshold Helpers

In [3]:
def safe_average_precision(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, scores))


def safe_roc_auc(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, scores))


def cost_components(y_true, y_pred, amount, alpha, beta):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    amount = np.asarray(amount, dtype=float)
    fn_mask = (y_true == 1) & (y_pred == 0)
    fp_mask = (y_true == 0) & (y_pred == 1)
    fn_cost = float(np.sum(amount[fn_mask] * beta))
    fp_cost = float(np.sum(amount[fp_mask] * alpha))
    return fn_cost, fp_cost, fn_cost + fp_cost


def row_cost(y_true, y_pred, amount, alpha, beta):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    amount = np.asarray(amount, dtype=float)
    return (
        ((y_true == 1) & (y_pred == 0)).astype(float) * amount * beta
        + ((y_true == 0) & (y_pred == 1)).astype(float) * amount * alpha
    )


def approve_all_cost(y_true, amount, beta):
    y_true = np.asarray(y_true, dtype=int)
    amount = np.asarray(amount, dtype=float)
    return float(np.sum(amount[y_true == 1] * beta))


def evaluate_predictions(policy_name, level, split_name, cost_config_name, y_true, scores, y_pred, amount, alpha, beta, extra=None):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    y_pred = np.asarray(y_pred, dtype=int)
    amount = np.asarray(amount, dtype=float)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fn_cost, fp_cost, total_cost = cost_components(y_true, y_pred, amount, alpha, beta)
    reference_cost = approve_all_cost(y_true, amount, beta)
    row = {
        "run_mode": RUN_MODE,
        "sample_rows": SAMPLE_ROWS_LABEL,
        "level": level,
        "policy": policy_name,
        "source_model": SELECTED_RISK_MODEL,
        "split": split_name,
        "cost_config": cost_config_name,
        "alpha": alpha,
        "beta": beta,
        "pr_auc": safe_average_precision(y_true, scores),
        "roc_auc": safe_roc_auc(y_true, scores),
        "recall_fraud": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision_fraud": float(precision_score(y_true, y_pred, zero_division=0)),
        "f1_fraud": float(f1_score(y_true, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fn_cost": fn_cost,
        "fp_cost": fp_cost,
        "total_cost": total_cost,
        "approve_all_cost": reference_cost,
        "cost_saving_vs_approve_all": reference_cost - total_cost,
    }
    if reference_cost:
        row["cost_saving_pct_vs_approve_all"] = (reference_cost - total_cost) / reference_cost
    else:
        row["cost_saving_pct_vs_approve_all"] = np.nan
    if extra:
        row.update(extra)
    return row


def threshold_curve(frame, cost_config_name, alpha, beta, threshold_grid=THRESHOLD_GRID):
    y = frame["isFraud"].to_numpy(dtype=int)
    scores = frame["score"].to_numpy(dtype=float)
    amount = frame["TransactionAmt"].to_numpy(dtype=float)
    rows = []
    for threshold in threshold_grid:
        pred = (scores >= threshold).astype(int)
        fn_cost, fp_cost, total_cost = cost_components(y, pred, amount, alpha, beta)
        rows.append({
            "cost_config": cost_config_name,
            "alpha": alpha,
            "beta": beta,
            "threshold": float(threshold),
            "fn_cost": fn_cost,
            "fp_cost": fp_cost,
            "total_cost": total_cost,
            "recall_fraud": float(recall_score(y, pred, zero_division=0)),
            "precision_fraud": float(precision_score(y, pred, zero_division=0)),
            "f1_fraud": float(f1_score(y, pred, zero_division=0)),
        })
    return pd.DataFrame(rows)


def select_curve_row(curve):
    return curve.sort_values(
        ["total_cost", "fp_cost", "recall_fraud", "precision_fraud", "threshold"],
        ascending=[True, True, False, False, False],
    ).iloc[0].to_dict()


def safe_id(*parts):
    text = "_".join(str(p) for p in parts)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    return text[:180]


def make_bin_table(strategy_name, source_amount):
    source_amount = pd.Series(source_amount, dtype="float64").replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    if len(source_amount) == 0:
        raise ValueError("Cannot build bins from empty amount source.")

    if strategy_name == "quantile_50_80_95":
        quantiles = [0.0, 0.50, 0.80, 0.95, 1.0]
        labels_base = ["low", "medium", "high", "very_high"]
        q_values = np.quantile(source_amount, quantiles)
        inner = []
        for value in q_values[1:-1]:
            value = float(value)
            if not inner or value > inner[-1]:
                inner.append(value)
        edges = [-np.inf, *inner, np.inf]
        labels = labels_base[:len(edges) - 1]
        policy = ",".join(map(str, quantiles))
    elif strategy_name == "quantile_60_85_97":
        quantiles = [0.0, 0.60, 0.85, 0.97, 1.0]
        labels_base = ["low", "medium", "high", "very_high"]
        q_values = np.quantile(source_amount, quantiles)
        inner = []
        for value in q_values[1:-1]:
            value = float(value)
            if not inner or value > inner[-1]:
                inner.append(value)
        edges = [-np.inf, *inner, np.inf]
        labels = labels_base[:len(edges) - 1]
        policy = ",".join(map(str, quantiles))
    elif strategy_name == "fixed_business_amount":
        edges = [-np.inf, 50.0, 150.0, 500.0, 2000.0, np.inf]
        labels = ["very_low", "low", "medium", "high", "very_high"]
        policy = "fixed:-inf,50,150,500,2000,inf"
    else:
        raise ValueError(f"Unknown bin strategy: {strategy_name}")

    return pd.DataFrame({
        "bin_strategy": strategy_name,
        "boundary_source": "validation_amounts_only",
        "amount_bin": labels,
        "bin_order": range(len(labels)),
        "bin_lower_bound": edges[:-1],
        "bin_upper_bound": edges[1:],
        "bin_policy": policy,
    })


def bin_table_from_phase31(amount_thresholds_df):
    first_cost = sorted(amount_thresholds_df["cost_config"].unique())[0]
    table = amount_thresholds_df[amount_thresholds_df["cost_config"] == first_cost].copy()
    table = table.sort_values(["bin_lower_bound", "bin_upper_bound"]).reset_index(drop=True)
    return pd.DataFrame({
        "bin_strategy": "phase31_quantile_50_80_95",
        "boundary_source": table["boundary_source_split"].astype(str).to_numpy(),
        "amount_bin": table["amount_bin"].astype(str).to_numpy(),
        "bin_order": range(len(table)),
        "bin_lower_bound": table["bin_lower_bound"].astype(float).to_numpy(),
        "bin_upper_bound": table["bin_upper_bound"].astype(float).to_numpy(),
        "bin_policy": "phase31_saved_boundaries",
    })


def assign_bins(amount, bin_table):
    table = bin_table.sort_values("bin_order").reset_index(drop=True)
    edges = list(table["bin_lower_bound"].astype(float)) + [float(table["bin_upper_bound"].iloc[-1])]
    labels = table["amount_bin"].astype(str).tolist()
    return pd.cut(
        pd.Series(amount, dtype="float64"),
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=True,
    ).astype(str).to_numpy()


def apply_threshold_table(frame, threshold_table):
    table = threshold_table.sort_values("bin_order").reset_index(drop=True)
    if set(table["amount_bin"].astype(str)) == {"all"}:
        thresholds = np.full(len(frame), float(table["threshold"].iloc[0]), dtype=float)
        amount_bins = np.array(["all"] * len(frame), dtype=object)
    else:
        amount_bins = assign_bins(frame["TransactionAmt"].to_numpy(dtype=float), table)
        lookup = dict(zip(table["amount_bin"].astype(str), table["threshold"].astype(float)))
        thresholds = pd.Series(amount_bins).map(lookup).astype(float).to_numpy()
        if np.isnan(thresholds).any():
            raise ValueError("Some rows did not map to a threshold.")
    pred = (frame["score"].to_numpy(dtype=float) >= thresholds).astype(int)
    return pred, thresholds, amount_bins


def evaluate_threshold_table(policy_name, candidate_id, threshold_table, frame, split_name, cost_config_name, alpha, beta, extra=None):
    pred, thresholds, amount_bins = apply_threshold_table(frame, threshold_table)
    details = {
        "candidate_id": candidate_id,
        "threshold_min": float(np.min(thresholds)) if len(thresholds) else np.nan,
        "threshold_max": float(np.max(thresholds)) if len(thresholds) else np.nan,
        "amount_bin_count": int(pd.Series(amount_bins).nunique()),
    }
    if extra:
        details.update(extra)
    return evaluate_predictions(
        policy_name=policy_name,
        level=details.pop("level", 4),
        split_name=split_name,
        cost_config_name=cost_config_name,
        y_true=frame["isFraud"].to_numpy(dtype=int),
        scores=frame["score"].to_numpy(dtype=float),
        y_pred=pred,
        amount=frame["TransactionAmt"].to_numpy(dtype=float),
        alpha=alpha,
        beta=beta,
        extra=details,
    )


def make_global_threshold_table(cost_config_name, alpha, beta, threshold, candidate_id, policy_name):
    return pd.DataFrame([{
        "run_mode": RUN_MODE,
        "sample_rows": SAMPLE_ROWS_LABEL,
        "source_model": SELECTED_RISK_MODEL,
        "cost_config": cost_config_name,
        "alpha": alpha,
        "beta": beta,
        "policy": policy_name,
        "candidate_id": candidate_id,
        "threshold_policy": "global_scalar",
        "bin_strategy": "global",
        "boundary_source": "not_applicable",
        "amount_bin": "all",
        "bin_order": 0,
        "bin_lower_bound": -np.inf,
        "bin_upper_bound": np.inf,
        "threshold": float(threshold),
        "validation_rows": len(validation_df),
        "validation_fraud_count": int(validation_df["isFraud"].sum()),
        "fallback_used": False,
        "fallback_reason": "",
    }])


def annotate_threshold_table(table, policy, candidate_id, cost_config_name, alpha, beta, threshold_policy, extra=None):
    out = table.copy()
    out["run_mode"] = RUN_MODE
    out["sample_rows"] = SAMPLE_ROWS_LABEL
    out["source_model"] = SELECTED_RISK_MODEL
    out["cost_config"] = cost_config_name
    out["alpha"] = alpha
    out["beta"] = beta
    out["policy"] = policy
    out["candidate_id"] = candidate_id
    out["threshold_policy"] = threshold_policy
    if extra:
        for key, value in extra.items():
            out[key] = value
    return out

## 3. Reconstruct Phase 3.1 References

In [4]:
phase31_bin_table = bin_table_from_phase31(amount_thresholds_df)
validation_df["amount_bin_phase31"] = assign_bins(validation_df["TransactionAmt"], phase31_bin_table)
test_df["amount_bin_phase31"] = assign_bins(test_df["TransactionAmt"], phase31_bin_table)

candidate_metric_rows = []
candidate_threshold_tables = []

# Level 3 global reference.
for cost_config_name, cfg in COST_CONFIGS.items():
    alpha, beta = cfg["alpha"], cfg["beta"]
    threshold = float(global_thresholds_df.loc[
        global_thresholds_df["cost_config"].eq(cost_config_name), "threshold"
    ].iloc[0])
    candidate_id = safe_id("global_reference", cost_config_name, threshold)
    table = make_global_threshold_table(
        cost_config_name, alpha, beta, threshold, candidate_id, "level3_global_reference"
    )
    candidate_threshold_tables.append(table)
    for split_name, frame in [("validation", validation_df), ("test", test_df)]:
        candidate_metric_rows.append(evaluate_threshold_table(
            policy_name="level3_global_reference",
            candidate_id=candidate_id,
            threshold_table=table,
            frame=frame,
            split_name=split_name,
            cost_config_name=cost_config_name,
            alpha=alpha,
            beta=beta,
            extra={
                "level": 3,
                "candidate_family": "reference",
                "selection_pool": False,
                "selection_status": "reference",
                "selector": "",
            },
        ))

# Phase 3.1 amount-bin Level 4 reference.
for cost_config_name, cfg in COST_CONFIGS.items():
    alpha, beta = cfg["alpha"], cfg["beta"]
    saved = amount_thresholds_df[amount_thresholds_df["cost_config"].eq(cost_config_name)].copy()
    saved = saved.sort_values(["bin_lower_bound", "bin_upper_bound"]).reset_index(drop=True)
    candidate_id = safe_id("phase31_level4_reference", cost_config_name)
    table = phase31_bin_table.copy()
    table["threshold"] = saved["threshold"].astype(float).to_numpy()
    table["validation_rows"] = saved["validation_rows"].astype(int).to_numpy()
    table["validation_fraud_count"] = saved["validation_fraud_count"].astype(int).to_numpy()
    table["fallback_used"] = saved["fallback_used"].astype(bool).to_numpy()
    table["fallback_reason"] = saved["fallback_reason"].fillna("").astype(str).to_numpy()
    table = annotate_threshold_table(
        table,
        policy="level4_amount_bin_dynamic_threshold_v1",
        candidate_id=candidate_id,
        cost_config_name=cost_config_name,
        alpha=alpha,
        beta=beta,
        threshold_policy="phase31_saved_amount_bin",
    )
    candidate_threshold_tables.append(table)
    for split_name, frame in [("validation", validation_df), ("test", test_df)]:
        candidate_metric_rows.append(evaluate_threshold_table(
            policy_name="level4_amount_bin_dynamic_threshold_v1",
            candidate_id=candidate_id,
            threshold_table=table,
            frame=frame,
            split_name=split_name,
            cost_config_name=cost_config_name,
            alpha=alpha,
            beta=beta,
            extra={
                "candidate_family": "reference",
                "selection_pool": True,
                "selection_status": "reference",
                "selector": "",
                "bin_strategy": "phase31_quantile_50_80_95",
            },
        ))

print("Phase 3.1 bin table")
display(phase31_bin_table)
print("Reference metrics")
display(pd.DataFrame(candidate_metric_rows)[pd.DataFrame(candidate_metric_rows)["split"].eq("test")][[
    "level", "policy", "cost_config", "recall_fraud", "precision_fraud",
    "fp_cost", "total_cost", "cost_saving_vs_approve_all"
]].sort_values(["cost_config", "level", "policy"]))

Phase 3.1 bin table


,bin_strategy,boundary_source,amount_bin,bin_order,bin_lower_bound,bin_upper_bound,bin_policy
0,phase31_quantile_50_80_95,validation,low,0,-inf,91.414001,phase31_saved_boundaries
1,phase31_quantile_50_80_95,validation,medium,1,91.414001,181.720395,phase31_saved_boundaries
2,phase31_quantile_50_80_95,validation,high,2,181.720395,461.835988,phase31_saved_boundaries
3,phase31_quantile_50_80_95,validation,very_high,3,461.835988,inf,phase31_saved_boundaries


Reference metrics


,level,policy,cost_config,recall_fraud,precision_fraud,fp_cost,total_cost,cost_saving_vs_approve_all
1,3,level3_global_reference,Cost-A,0.567213,0.335271,2159.069448,20198.387418,17244.647541
7,4,level4_amount_bin_dynamic_threshold_v1,Cost-A,0.655738,0.148368,4402.340947,21028.004927,16415.030032
3,3,level3_global_reference,Cost-B,0.567213,0.335271,4318.138897,40396.774837,34489.295082
9,4,level4_amount_bin_dynamic_threshold_v1,Cost-B,0.655738,0.148368,8804.681893,42056.009854,32830.060064
5,3,level3_global_reference,Cost-C,0.731148,0.098716,57246.004751,109064.429634,78150.745163
11,4,level4_amount_bin_dynamic_threshold_v1,Cost-C,0.714754,0.111623,38720.450946,102499.135839,84716.038958


## 4. Candidate 1 - Shrink Per-Bin Thresholds Toward Global

In [5]:
for cost_config_name, cfg in COST_CONFIGS.items():
    alpha, beta = cfg["alpha"], cfg["beta"]
    global_threshold = float(global_thresholds_df.loc[
        global_thresholds_df["cost_config"].eq(cost_config_name), "threshold"
    ].iloc[0])
    saved = amount_thresholds_df[amount_thresholds_df["cost_config"].eq(cost_config_name)].copy()
    saved = saved.sort_values(["bin_lower_bound", "bin_upper_bound"]).reset_index(drop=True)
    original_thresholds = saved["threshold"].astype(float).to_numpy()

    for lambda_value in LAMBDA_GRID:
        candidate_id = safe_id("shrunk", cost_config_name, f"lambda_{lambda_value:.2f}")
        table = phase31_bin_table.copy()
        table["threshold"] = lambda_value * original_thresholds + (1.0 - lambda_value) * global_threshold
        table["validation_rows"] = saved["validation_rows"].astype(int).to_numpy()
        table["validation_fraud_count"] = saved["validation_fraud_count"].astype(int).to_numpy()
        table["fallback_used"] = False
        table["fallback_reason"] = ""
        table = annotate_threshold_table(
            table,
            policy="level4_shrunk_amount_bin_threshold",
            candidate_id=candidate_id,
            cost_config_name=cost_config_name,
            alpha=alpha,
            beta=beta,
            threshold_policy="shrink_to_global",
            extra={"lambda_value": lambda_value, "bin_strategy": "phase31_quantile_50_80_95"},
        )
        candidate_threshold_tables.append(table)
        candidate_metric_rows.append(evaluate_threshold_table(
            policy_name="level4_shrunk_amount_bin_threshold",
            candidate_id=candidate_id,
            threshold_table=table,
            frame=validation_df,
            split_name="validation",
            cost_config_name=cost_config_name,
            alpha=alpha,
            beta=beta,
            extra={
                "candidate_family": "shrinkage",
                "selection_pool": True,
                "selection_status": "candidate_validation",
                "selector": "",
                "lambda_value": lambda_value,
                "bin_strategy": "phase31_quantile_50_80_95",
            },
        ))

shrink_preview = pd.DataFrame(candidate_metric_rows)
print("Shrinkage candidates on validation")
display(shrink_preview[
    shrink_preview["policy"].eq("level4_shrunk_amount_bin_threshold")
][[
    "cost_config", "candidate_id", "lambda_value", "recall_fraud",
    "precision_fraud", "fp_cost", "total_cost"
]].sort_values(["cost_config", "total_cost"]).head(20))

Shrinkage candidates on validation


,cost_config,candidate_id,lambda_value,recall_fraud,precision_fraud,fp_cost,total_cost
16,Cost-A,shrunk_cost_a_lambda_1_00,1.00,0.761273,0.243014,3757.895798,20395.969750
15,Cost-A,shrunk_cost_a_lambda_0_75,0.75,0.726790,0.302428,3125.316899,20635.042850
14,Cost-A,shrunk_cost_a_lambda_0_50,0.50,0.689655,0.354707,2696.005250,21200.207211
13,Cost-A,shrunk_cost_a_lambda_0_25,0.25,0.639257,0.393791,2484.389251,22246.685213
12,Cost-A,shrunk_cost_a_lambda_0_00,0.00,0.615385,0.483333,2195.424150,22423.988114
21,Cost-B,shrunk_cost_b_lambda_1_00,1.00,0.761273,0.243014,7515.791597,40791.939500
20,Cost-B,shrunk_cost_b_lambda_0_75,0.75,0.726790,0.302428,6250.633799,41270.085699
19,Cost-B,shrunk_cost_b_lambda_0_50,0.50,0.689655,0.354707,5392.010500,42400.414423
18,Cost-B,shrunk_cost_b_lambda_0_25,0.25,0.639257,0.393791,4968.778501,44493.370427
17,Cost-B,shrunk_cost_b_lambda_0_00,0.00,0.615385,0.483333,4390.848300,44847.976228


## 5. Candidate 2 - Precision / FP-Cost Guard

In [6]:
def tune_guarded_threshold_table(cost_config_name, alpha, beta, precision_floor, fp_cost_cap_multiplier):
    global_threshold = float(global_thresholds_df.loc[
        global_thresholds_df["cost_config"].eq(cost_config_name), "threshold"
    ].iloc[0])
    candidate_id = safe_id(
        "precision_guard",
        cost_config_name,
        f"precision_{precision_floor:.2f}",
        f"fpcap_{fp_cost_cap_multiplier:.2f}",
    )
    rows = []
    for _, bin_row in phase31_bin_table.iterrows():
        amount_bin = str(bin_row["amount_bin"])
        bin_mask = validation_df["amount_bin_phase31"].astype(str).eq(amount_bin).to_numpy()
        frame_bin = validation_df.loc[bin_mask].copy()
        rows_count = int(len(frame_bin))
        fraud_count = int(frame_bin["isFraud"].sum())

        fallback_used = rows_count < MIN_BIN_VALIDATION_ROWS or fraud_count < MIN_BIN_VALIDATION_FRAUD
        fallback_reason = ""
        selected = None

        if not fallback_used:
            global_pred = (frame_bin["score"].to_numpy(dtype=float) >= global_threshold).astype(int)
            _, global_fp_cost, _ = cost_components(
                frame_bin["isFraud"].to_numpy(dtype=int),
                global_pred,
                frame_bin["TransactionAmt"].to_numpy(dtype=float),
                alpha,
                beta,
            )
            fp_cap = global_fp_cost * fp_cost_cap_multiplier
            curve = threshold_curve(frame_bin, cost_config_name, alpha, beta)
            feasible = curve[
                (curve["precision_fraud"] >= precision_floor)
                & (curve["fp_cost"] <= fp_cap + 1e-9)
            ].copy()
            if feasible.empty:
                fallback_used = True
                fallback_reason = "no_threshold_meets_precision_fp_guard"
            else:
                selected = select_curve_row(feasible)

        if fallback_used:
            if rows_count < MIN_BIN_VALIDATION_ROWS:
                fallback_reason = f"rows<{MIN_BIN_VALIDATION_ROWS}"
            elif fraud_count < MIN_BIN_VALIDATION_FRAUD:
                fallback_reason = f"fraud<{MIN_BIN_VALIDATION_FRAUD}"
            threshold = global_threshold
        else:
            threshold = float(selected["threshold"])

        rows.append({
            **bin_row.to_dict(),
            "threshold": threshold,
            "validation_rows": rows_count,
            "validation_fraud_count": fraud_count,
            "fallback_used": bool(fallback_used),
            "fallback_reason": fallback_reason,
        })

    table = pd.DataFrame(rows)
    return annotate_threshold_table(
        table,
        policy="level4_precision_guard_threshold",
        candidate_id=candidate_id,
        cost_config_name=cost_config_name,
        alpha=alpha,
        beta=beta,
        threshold_policy="precision_fp_guard",
        extra={
            "precision_floor": precision_floor,
            "fp_cost_cap_multiplier": fp_cost_cap_multiplier,
            "bin_strategy": "phase31_quantile_50_80_95",
        },
    )


for cost_config_name, cfg in COST_CONFIGS.items():
    alpha, beta = cfg["alpha"], cfg["beta"]
    for precision_floor in PRECISION_FLOOR_GRID:
        for fp_cap_multiplier in FP_COST_CAP_MULTIPLIER_GRID:
            table = tune_guarded_threshold_table(cost_config_name, alpha, beta, precision_floor, fp_cap_multiplier)
            candidate_threshold_tables.append(table)
            candidate_metric_rows.append(evaluate_threshold_table(
                policy_name="level4_precision_guard_threshold",
                candidate_id=str(table["candidate_id"].iloc[0]),
                threshold_table=table,
                frame=validation_df,
                split_name="validation",
                cost_config_name=cost_config_name,
                alpha=alpha,
                beta=beta,
                extra={
                    "candidate_family": "precision_guard",
                    "selection_pool": True,
                    "selection_status": "candidate_validation",
                    "selector": "",
                    "precision_floor": precision_floor,
                    "fp_cost_cap_multiplier": fp_cap_multiplier,
                    "bin_strategy": "phase31_quantile_50_80_95",
                    "fallback_used": bool(table["fallback_used"].any()),
                },
            ))

guard_preview = pd.DataFrame(candidate_metric_rows)
print("Precision/FP guard candidates on validation")
display(guard_preview[
    guard_preview["policy"].eq("level4_precision_guard_threshold")
][[
    "cost_config", "candidate_id", "precision_floor", "fp_cost_cap_multiplier",
    "recall_fraud", "precision_fraud", "fp_cost", "total_cost"
]].sort_values(["cost_config", "total_cost"]).head(20))

Precision/FP guard candidates on validation


,cost_config,candidate_id,precision_floor,fp_cost_cap_multiplier,recall_fraud,precision_fraud,fp_cost,total_cost
30,Cost-A,precision_guard_cost_a_precision_0_05_fpcap_2_00,0.05,2.00,0.647215,0.392915,2449.16925,21704.031213
34,Cost-A,precision_guard_cost_a_precision_0_10_fpcap_2_00,0.10,2.00,0.647215,0.392915,2449.16925,21704.031213
38,Cost-A,precision_guard_cost_a_precision_0_15_fpcap_2_00,0.15,2.00,0.647215,0.392915,2449.16925,21704.031213
29,Cost-A,precision_guard_cost_a_precision_0_05_fpcap_1_50,0.05,1.50,0.631300,0.466667,2146.46950,21783.033464
33,Cost-A,precision_guard_cost_a_precision_0_10_fpcap_1_50,0.10,1.50,0.631300,0.466667,2146.46950,21783.033464
37,Cost-A,precision_guard_cost_a_precision_0_15_fpcap_1_50,0.15,1.50,0.631300,0.466667,2146.46950,21783.033464
28,Cost-A,precision_guard_cost_a_precision_0_05_fpcap_1_25,0.05,1.25,0.623342,0.473790,2081.08005,22067.644014
32,Cost-A,precision_guard_cost_a_precision_0_10_fpcap_1_25,0.10,1.25,0.623342,0.473790,2081.08005,22067.644014
36,Cost-A,precision_guard_cost_a_precision_0_15_fpcap_1_25,0.15,1.25,0.623342,0.473790,2081.08005,22067.644014
27,Cost-A,precision_guard_cost_a_precision_0_05_fpcap_1_00,0.05,1.00,0.615385,0.498925,1972.42375,22200.987714


## 6. Candidate 3 - Amount-Bin Strategy Grid

In [7]:
BIN_STRATEGIES = ["quantile_50_80_95", "quantile_60_85_97", "fixed_business_amount"]


def tune_per_bin_threshold_table(bin_table, cost_config_name, alpha, beta, policy_name, candidate_id):
    global_threshold = float(global_thresholds_df.loc[
        global_thresholds_df["cost_config"].eq(cost_config_name), "threshold"
    ].iloc[0])
    validation_bins = assign_bins(validation_df["TransactionAmt"], bin_table)
    rows = []
    for _, bin_row in bin_table.iterrows():
        amount_bin = str(bin_row["amount_bin"])
        bin_mask = validation_bins == amount_bin
        frame_bin = validation_df.loc[bin_mask].copy()
        rows_count = int(len(frame_bin))
        fraud_count = int(frame_bin["isFraud"].sum())

        fallback_used = rows_count < MIN_BIN_VALIDATION_ROWS or fraud_count < MIN_BIN_VALIDATION_FRAUD
        fallback_reason = ""
        selected = None

        if not fallback_used:
            curve = threshold_curve(frame_bin, cost_config_name, alpha, beta)
            selected = select_curve_row(curve)

        if fallback_used:
            if rows_count < MIN_BIN_VALIDATION_ROWS:
                fallback_reason = f"rows<{MIN_BIN_VALIDATION_ROWS}"
            elif fraud_count < MIN_BIN_VALIDATION_FRAUD:
                fallback_reason = f"fraud<{MIN_BIN_VALIDATION_FRAUD}"
            threshold = global_threshold
        else:
            threshold = float(selected["threshold"])

        rows.append({
            **bin_row.to_dict(),
            "threshold": threshold,
            "validation_rows": rows_count,
            "validation_fraud_count": fraud_count,
            "fallback_used": bool(fallback_used),
            "fallback_reason": fallback_reason,
        })

    table = pd.DataFrame(rows)
    return annotate_threshold_table(
        table,
        policy=policy_name,
        candidate_id=candidate_id,
        cost_config_name=cost_config_name,
        alpha=alpha,
        beta=beta,
        threshold_policy="bin_strategy_grid",
        extra={"bin_strategy": str(bin_table["bin_strategy"].iloc[0])},
    )


for strategy_name in BIN_STRATEGIES:
    bin_table = make_bin_table(strategy_name, validation_df["TransactionAmt"])
    for cost_config_name, cfg in COST_CONFIGS.items():
        alpha, beta = cfg["alpha"], cfg["beta"]
        candidate_id = safe_id("bin_strategy", strategy_name, cost_config_name)
        table = tune_per_bin_threshold_table(
            bin_table=bin_table,
            cost_config_name=cost_config_name,
            alpha=alpha,
            beta=beta,
            policy_name="level4_bin_strategy_grid",
            candidate_id=candidate_id,
        )
        candidate_threshold_tables.append(table)
        candidate_metric_rows.append(evaluate_threshold_table(
            policy_name="level4_bin_strategy_grid",
            candidate_id=candidate_id,
            threshold_table=table,
            frame=validation_df,
            split_name="validation",
            cost_config_name=cost_config_name,
            alpha=alpha,
            beta=beta,
            extra={
                "candidate_family": "bin_strategy_grid",
                "selection_pool": True,
                "selection_status": "candidate_validation",
                "selector": "",
                "bin_strategy": strategy_name,
                "fallback_used": bool(table["fallback_used"].any()),
            },
        ))

bin_grid_preview = pd.DataFrame(candidate_metric_rows)
print("Bin strategy candidates on validation")
display(bin_grid_preview[
    bin_grid_preview["policy"].eq("level4_bin_strategy_grid")
][[
    "cost_config", "candidate_id", "bin_strategy", "recall_fraud",
    "precision_fraud", "fp_cost", "total_cost"
]].sort_values(["cost_config", "total_cost"]))

Bin strategy candidates on validation


,cost_config,candidate_id,bin_strategy,recall_fraud,precision_fraud,fp_cost,total_cost
90,Cost-A,bin_strategy_quantile_60_85_97_cost_a,quantile_60_85_97,0.742706,0.278607,5857.831739,19968.487743
87,Cost-A,bin_strategy_quantile_50_80_95_cost_a,quantile_50_80_95,0.761273,0.243014,3757.895798,20395.969750
93,Cost-A,bin_strategy_fixed_business_amount_cost_a,fixed_business_amount,0.769231,0.231629,7649.861992,20634.617994
91,Cost-B,bin_strategy_quantile_60_85_97_cost_b,quantile_60_85_97,0.742706,0.278607,11715.663479,39936.975486
88,Cost-B,bin_strategy_quantile_50_80_95_cost_b,quantile_50_80_95,0.761273,0.243014,7515.791597,40791.939500
94,Cost-B,bin_strategy_fixed_business_amount_cost_b,fixed_business_amount,0.769231,0.231629,15299.723983,41269.235988
92,Cost-C,bin_strategy_quantile_60_85_97_cost_c,quantile_60_85_97,0.814324,0.182196,33608.771143,93330.006170
95,Cost-C,bin_strategy_fixed_business_amount_cost_c,fixed_business_amount,0.779841,0.219567,31961.818355,96135.818370
89,Cost-C,bin_strategy_quantile_50_80_95_cost_c,quantile_50_80_95,0.816976,0.181176,37279.524352,96809.399363


## 7. Validation-Only Selection and Frozen Test Evaluation

In [8]:
candidate_metrics_df = pd.DataFrame(candidate_metric_rows)
candidate_thresholds_df = pd.concat(candidate_threshold_tables, ignore_index=True, sort=False)

POLICY_PRIORITY = {
    "level3_global_reference": 0,
    "level4_shrunk_amount_bin_threshold": 1,
    "level4_amount_bin_dynamic_threshold_v1": 2,
    "level4_precision_guard_threshold": 3,
    "level4_bin_strategy_grid": 4,
}


def rank_candidates(df):
    out = df.copy()
    out["policy_priority"] = out["policy"].map(POLICY_PRIORITY).fillna(99)
    return out.sort_values(
        ["total_cost", "fp_cost", "recall_fraud", "precision_fraud", "policy_priority"],
        ascending=[True, True, False, False, True],
    )


def select_best_total(cost_config_name):
    pool = candidate_metrics_df[
        candidate_metrics_df["split"].eq("validation")
        & candidate_metrics_df["cost_config"].eq(cost_config_name)
        & candidate_metrics_df["selection_pool"].astype(bool)
    ].copy()
    return rank_candidates(pool).iloc[0]


def select_guarded_tradeoff(cost_config_name):
    pool = candidate_metrics_df[
        candidate_metrics_df["split"].eq("validation")
        & candidate_metrics_df["cost_config"].eq(cost_config_name)
        & candidate_metrics_df["selection_pool"].astype(bool)
    ].copy()
    best_total = float(pool["total_cost"].min())
    eligible = pool[pool["total_cost"] <= best_total * GUARDED_SELECTOR_TOLERANCE].copy()

    if cost_config_name in {"Cost-A", "Cost-B"}:
        original = pool[pool["policy"].eq("level4_amount_bin_dynamic_threshold_v1")]
        if not original.empty:
            original_fp = float(original["fp_cost"].iloc[0])
            fp_guarded = eligible[eligible["fp_cost"] <= original_fp + 1e-9].copy()
            if not fp_guarded.empty:
                eligible = fp_guarded
        eligible["policy_priority"] = eligible["policy"].map(POLICY_PRIORITY).fillna(99)
        return eligible.sort_values(
            ["fp_cost", "precision_fraud", "total_cost", "recall_fraud", "policy_priority"],
            ascending=[True, False, True, False, True],
        ).iloc[0]

    return rank_candidates(eligible).iloc[0]


def threshold_table_for_candidate(candidate_id, cost_config_name):
    table = candidate_thresholds_df[
        candidate_thresholds_df["candidate_id"].astype(str).eq(str(candidate_id))
        & candidate_thresholds_df["cost_config"].eq(cost_config_name)
    ].copy()
    if table.empty:
        raise ValueError(f"No threshold table for candidate_id={candidate_id}, cost={cost_config_name}")
    return table.sort_values("bin_order").reset_index(drop=True)


selected_rows = []
selected_metric_rows = []

for selector_name, selector_func in [
    ("level4_tuned_best_cost_selector", select_best_total),
    ("level4_tuned_guarded_selector", select_guarded_tradeoff),
]:
    for cost_config_name, cfg in COST_CONFIGS.items():
        alpha, beta = cfg["alpha"], cfg["beta"]
        selected = selector_func(cost_config_name)
        candidate_id = str(selected["candidate_id"])
        table = threshold_table_for_candidate(candidate_id, cost_config_name)

        selected_info = selected.to_dict()
        selected_info.update({
            "selector": selector_name,
            "selected_candidate_id": candidate_id,
            "selected_candidate_policy": selected["policy"],
            "selected_on": "validation",
            "selector_tolerance": GUARDED_SELECTOR_TOLERANCE if selector_name.endswith("guarded_selector") else np.nan,
        })
        selected_rows.append(selected_info)

        for split_name, frame in [("validation", validation_df), ("test", test_df)]:
            selected_metric_rows.append(evaluate_threshold_table(
                policy_name=selector_name,
                candidate_id=candidate_id,
                threshold_table=table,
                frame=frame,
                split_name=split_name,
                cost_config_name=cost_config_name,
                alpha=alpha,
                beta=beta,
                extra={
                    "candidate_family": "selected_policy",
                    "selection_pool": False,
                    "selection_status": "selected_validation" if split_name == "validation" else "selected_test",
                    "selector": selector_name,
                    "selected_candidate_policy": selected["policy"],
                    "selected_candidate_id": candidate_id,
                    "selected_validation_total_cost": float(selected["total_cost"]),
                    "selected_validation_fp_cost": float(selected["fp_cost"]),
                    "bin_strategy": str(table["bin_strategy"].iloc[0]),
                    "fallback_used": bool(table["fallback_used"].any()),
                },
            ))

selected_policies_df = pd.DataFrame(selected_rows)
selected_metrics_df = pd.DataFrame(selected_metric_rows)

print("Selected policies from validation")
display(selected_policies_df[[
    "selector", "cost_config", "selected_candidate_policy", "selected_candidate_id",
    "total_cost", "fp_cost", "recall_fraud", "precision_fraud"
]].sort_values(["selector", "cost_config"]))

print("Frozen selected-policy test metrics")
display(selected_metrics_df[selected_metrics_df["split"].eq("test")][[
    "selector", "policy", "cost_config", "selected_candidate_policy",
    "recall_fraud", "precision_fraud", "fp_cost", "total_cost", "cost_saving_vs_approve_all"
]].sort_values(["selector", "cost_config"]))

Selected policies from validation


,selector,cost_config,selected_candidate_policy,selected_candidate_id,total_cost,fp_cost,recall_fraud,precision_fraud
0,level4_tuned_best_cost_selector,Cost-A,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_a,19968.487743,5857.831739,0.742706,0.278607
1,level4_tuned_best_cost_selector,Cost-B,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_b,39936.975486,11715.663479,0.742706,0.278607
2,level4_tuned_best_cost_selector,Cost-C,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_c,93330.006170,33608.771143,0.814324,0.182196
3,level4_tuned_guarded_selector,Cost-A,level4_shrunk_amount_bin_threshold,shrunk_cost_a_lambda_0_75,20635.042850,3125.316899,0.726790,0.302428
4,level4_tuned_guarded_selector,Cost-B,level4_shrunk_amount_bin_threshold,shrunk_cost_b_lambda_0_75,41270.085699,6250.633799,0.726790,0.302428
5,level4_tuned_guarded_selector,Cost-C,level4_bin_strategy_grid,bin_strategy_quantile_60_85_97_cost_c,93330.006170,33608.771143,0.814324,0.182196


Frozen selected-policy test metrics


,selector,policy,cost_config,selected_candidate_policy,recall_fraud,precision_fraud,fp_cost,total_cost,cost_saving_vs_approve_all
1,level4_tuned_best_cost_selector,level4_tuned_best_cost_selector,Cost-A,level4_bin_strategy_grid,0.659016,0.179304,6041.124241,20048.006220,17395.028739
3,level4_tuned_best_cost_selector,level4_tuned_best_cost_selector,Cost-B,level4_bin_strategy_grid,0.659016,0.179304,12082.248482,40096.012441,34790.057478
5,level4_tuned_best_cost_selector,level4_tuned_best_cost_selector,Cost-C,level4_bin_strategy_grid,0.711475,0.113791,35830.219749,99834.099649,87381.075147
7,level4_tuned_guarded_selector,level4_tuned_guarded_selector,Cost-A,level4_shrunk_amount_bin_threshold,0.645902,0.189060,3653.052798,20332.934777,17110.100182
9,level4_tuned_guarded_selector,level4_tuned_guarded_selector,Cost-B,level4_shrunk_amount_bin_threshold,0.645902,0.189060,7306.105596,40665.869555,34220.200364
11,level4_tuned_guarded_selector,level4_tuned_guarded_selector,Cost-C,level4_bin_strategy_grid,0.711475,0.113791,35830.219749,99834.099649,87381.075147


## 8. Disagreement Analysis vs Phase 3.1 Level 4

In [9]:
disagreement_rows = []

for cost_config_name, cfg in COST_CONFIGS.items():
    alpha, beta = cfg["alpha"], cfg["beta"]
    original_candidate_id = safe_id("phase31_level4_reference", cost_config_name)
    original_table = threshold_table_for_candidate(original_candidate_id, cost_config_name)
    original_pred, _, _ = apply_threshold_table(test_df, original_table)
    original_cost = row_cost(
        test_df["isFraud"].to_numpy(dtype=int),
        original_pred,
        test_df["TransactionAmt"].to_numpy(dtype=float),
        alpha,
        beta,
    )

    guarded = selected_policies_df[
        selected_policies_df["selector"].eq("level4_tuned_guarded_selector")
        & selected_policies_df["cost_config"].eq(cost_config_name)
    ].iloc[0]
    tuned_table = threshold_table_for_candidate(str(guarded["selected_candidate_id"]), cost_config_name)
    tuned_pred, _, tuned_bins = apply_threshold_table(test_df, tuned_table)
    tuned_cost = row_cost(
        test_df["isFraud"].to_numpy(dtype=int),
        tuned_pred,
        test_df["TransactionAmt"].to_numpy(dtype=float),
        alpha,
        beta,
    )

    diff_mask = original_pred != tuned_pred
    if not np.any(diff_mask):
        continue
    diff = test_df.loc[diff_mask, [
        "TransactionID", "isFraud", "TransactionAmt", "score"
    ]].copy()
    diff["cost_config"] = cost_config_name
    diff["alpha"] = alpha
    diff["beta"] = beta
    diff["phase31_level4_action"] = original_pred[diff_mask]
    diff["phase31b_guarded_action"] = tuned_pred[diff_mask]
    diff["phase31b_amount_bin"] = np.asarray(tuned_bins)[diff_mask]
    diff["phase31_level4_cost"] = original_cost[diff_mask]
    diff["phase31b_guarded_cost"] = tuned_cost[diff_mask]
    diff["cost_delta_guarded_minus_phase31"] = diff["phase31b_guarded_cost"] - diff["phase31_level4_cost"]
    diff["selected_candidate_policy"] = guarded["selected_candidate_policy"]
    disagreement_rows.append(diff)

if disagreement_rows:
    disagreement_cases_df = pd.concat(disagreement_rows, ignore_index=True)
    disagreement_cases_df = disagreement_cases_df.sort_values(
        ["cost_config", "cost_delta_guarded_minus_phase31", "TransactionAmt"],
        ascending=[True, True, False],
    )
else:
    disagreement_cases_df = pd.DataFrame(columns=[
        "TransactionID", "isFraud", "TransactionAmt", "score", "cost_config",
        "phase31_level4_action", "phase31b_guarded_action",
        "cost_delta_guarded_minus_phase31",
    ])

print("Disagreement cases preview: negative cost_delta means Phase 3.1b guarded improved row cost")
display(disagreement_cases_df.head(30))

Disagreement cases preview: negative cost_delta means Phase 3.1b guarded improved row cost


,TransactionID,isFraud,TransactionAmt,score,cost_config,alpha,beta,phase31_level4_action,phase31b_guarded_action,phase31b_amount_bin,phase31_level4_cost,phase31b_guarded_cost,cost_delta_guarded_minus_phase31,selected_candidate_policy
280,3085988,0,167.000000,0.643541,Cost-A,0.05,1.0,1,0,medium,8.3500,0.0,-8.3500,level4_shrunk_amount_bin_threshold
99,3077180,0,150.000000,0.636204,Cost-A,0.05,1.0,1,0,medium,7.5000,0.0,-7.5000,level4_shrunk_amount_bin_threshold
266,3085449,0,150.000000,0.643275,Cost-A,0.05,1.0,1,0,medium,7.5000,0.0,-7.5000,level4_shrunk_amount_bin_threshold
256,3085107,0,146.000000,0.642728,Cost-A,0.05,1.0,1,0,medium,7.3000,0.0,-7.3000,level4_shrunk_amount_bin_threshold
285,3086202,0,135.255997,0.631760,Cost-A,0.05,1.0,1,0,medium,6.7628,0.0,-6.7628,level4_shrunk_amount_bin_threshold
182,3080708,0,131.000000,0.638554,Cost-A,0.05,1.0,1,0,medium,6.5500,0.0,-6.5500,level4_shrunk_amount_bin_threshold
67,3075183,0,130.500000,0.628100,Cost-A,0.05,1.0,1,0,medium,6.5250,0.0,-6.5250,level4_shrunk_amount_bin_threshold
204,3082020,0,130.500000,0.633158,Cost-A,0.05,1.0,1,0,medium,6.5250,0.0,-6.5250,level4_shrunk_amount_bin_threshold
189,3080966,0,125.000000,0.627352,Cost-A,0.05,1.0,1,0,medium,6.2500,0.0,-6.2500,level4_shrunk_amount_bin_threshold
83,3076140,0,119.632004,0.641450,Cost-A,0.05,1.0,1,0,medium,5.9816,0.0,-5.9816,level4_shrunk_amount_bin_threshold


## 9. Save Outputs and Five-Level Tuned Comparison

In [10]:
candidate_metrics_all_df = pd.concat([candidate_metrics_df, selected_metrics_df], ignore_index=True, sort=False)

selected_candidate_ids = set(selected_policies_df["selected_candidate_id"].astype(str))
candidate_thresholds_df["selected_by_phase31b"] = candidate_thresholds_df["candidate_id"].astype(str).isin(selected_candidate_ids)
candidate_thresholds_df["selected_by_best_cost"] = candidate_thresholds_df["candidate_id"].astype(str).isin(
    set(selected_policies_df[selected_policies_df["selector"].eq("level4_tuned_best_cost_selector")]["selected_candidate_id"].astype(str))
)
candidate_thresholds_df["selected_by_guarded"] = candidate_thresholds_df["candidate_id"].astype(str).isin(
    set(selected_policies_df[selected_policies_df["selector"].eq("level4_tuned_guarded_selector")]["selected_candidate_id"].astype(str))
)

CANDIDATE_METRICS_PATH = RESULTS_DIR / f"phase31b_candidate_metrics_{RUN_OUTPUT_TAG}.csv"
CANDIDATE_THRESHOLDS_PATH = RESULTS_DIR / f"phase31b_candidate_thresholds_{RUN_OUTPUT_TAG}.csv"
SELECTED_POLICIES_PATH = RESULTS_DIR / f"phase31b_selected_policies_{RUN_OUTPUT_TAG}.csv"
DISAGREEMENT_PATH = RESULTS_DIR / f"phase31b_disagreement_cases_{RUN_OUTPUT_TAG}.csv"
TUNED_COMPARISON_PATH = RESULTS_DIR / f"five_level_comparison_tuned_{RUN_OUTPUT_TAG}.csv"

candidate_metrics_all_df.to_csv(RESULTS_DIR / "phase31b_candidate_metrics.csv", index=False)
candidate_metrics_all_df.to_csv(CANDIDATE_METRICS_PATH, index=False)
candidate_thresholds_df.to_csv(RESULTS_DIR / "phase31b_candidate_thresholds.csv", index=False)
candidate_thresholds_df.to_csv(CANDIDATE_THRESHOLDS_PATH, index=False)
selected_policies_df.to_csv(RESULTS_DIR / "phase31b_selected_policies.csv", index=False)
selected_policies_df.to_csv(SELECTED_POLICIES_PATH, index=False)
disagreement_cases_df.to_csv(RESULTS_DIR / "phase31b_disagreement_cases.csv", index=False)
disagreement_cases_df.to_csv(DISAGREEMENT_PATH, index=False)

selected_test_for_comparison = selected_metrics_df[selected_metrics_df["split"].eq("test")].copy()
selected_test_for_comparison["model"] = selected_test_for_comparison["policy"]
selected_test_for_comparison["status"] = "complete_tuned_phase31b"
selected_test_for_comparison["level"] = 4
selected_test_for_comparison["level_label"] = "Tuned Level 4"

if not five_level_df.empty:
    five_level_tuned_df = pd.concat([five_level_df, selected_test_for_comparison], ignore_index=True, sort=False)
else:
    five_level_tuned_df = selected_test_for_comparison.copy()

five_level_tuned_df.to_csv(RESULTS_DIR / "five_level_comparison_tuned.csv", index=False)
five_level_tuned_df.to_csv(TUNED_COMPARISON_PATH, index=False)

print("Saved:")
for path in [
    CANDIDATE_METRICS_PATH,
    CANDIDATE_THRESHOLDS_PATH,
    SELECTED_POLICIES_PATH,
    DISAGREEMENT_PATH,
    TUNED_COMPARISON_PATH,
]:
    print("-", path)

summary_cols = [
    "policy", "cost_config", "selected_candidate_policy", "recall_fraud",
    "precision_fraud", "fp_cost", "total_cost", "cost_saving_vs_approve_all"
]
print("Phase 3.1b selected test summary")
display(selected_test_for_comparison[summary_cols].sort_values(["policy", "cost_config"]))

Saved:
- /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/phase31b_candidate_metrics_sample_100k.csv
- /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/phase31b_candidate_thresholds_sample_100k.csv
- /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/phase31b_selected_policies_sample_100k.csv
- /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/phase31b_disagreement_cases_sample_100k.csv
- /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/results/five_level_comparison_tuned_sample_100k.csv
Phase 3.1b selected test summary


,policy,cost_config,selected_candidate_policy,recall_fraud,precision_fraud,fp_cost,total_cost,cost_saving_vs_approve_all
1,level4_tuned_best_cost_selector,Cost-A,level4_bin_strategy_grid,0.659016,0.179304,6041.124241,20048.006220,17395.028739
3,level4_tuned_best_cost_selector,Cost-B,level4_bin_strategy_grid,0.659016,0.179304,12082.248482,40096.012441,34790.057478
5,level4_tuned_best_cost_selector,Cost-C,level4_bin_strategy_grid,0.711475,0.113791,35830.219749,99834.099649,87381.075147
7,level4_tuned_guarded_selector,Cost-A,level4_shrunk_amount_bin_threshold,0.645902,0.189060,3653.052798,20332.934777,17110.100182
9,level4_tuned_guarded_selector,Cost-B,level4_shrunk_amount_bin_threshold,0.645902,0.189060,7306.105596,40665.869555,34220.200364
11,level4_tuned_guarded_selector,Cost-C,level4_bin_strategy_grid,0.711475,0.113791,35830.219749,99834.099649,87381.075147


## 10. Figures

In [11]:
def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()


plot_refs = candidate_metrics_all_df[
    candidate_metrics_all_df["split"].eq("test")
    & candidate_metrics_all_df["policy"].isin([
        "level3_global_reference",
        "level4_amount_bin_dynamic_threshold_v1",
        "level4_tuned_best_cost_selector",
        "level4_tuned_guarded_selector",
    ])
].copy()

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_refs, x="cost_config", y="total_cost", hue="policy")
plt.title("Phase 3.1b Total Cost: Reference vs Tuned Selectors")
plt.xlabel("Cost configuration")
plt.ylabel("Total Cost")
plt.xticks(rotation=0)
save_fig(FIGURES_DIR / f"phase31b_total_cost_{RUN_OUTPUT_TAG}.png")

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_refs, x="cost_config", y="cost_saving_vs_approve_all", hue="policy")
plt.title("Phase 3.1b Cost Saving vs Approve-All")
plt.xlabel("Cost configuration")
plt.ylabel("Cost Saving")
plt.xticks(rotation=0)
save_fig(FIGURES_DIR / f"phase31b_cost_saving_{RUN_OUTPUT_TAG}.png")

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=plot_refs,
    x="fp_cost",
    y="precision_fraud",
    hue="policy",
    style="cost_config",
    s=120,
)
plt.title("Phase 3.1b FP Cost vs Fraud Precision")
plt.xlabel("False Positive Cost")
plt.ylabel("Fraud Precision")
save_fig(FIGURES_DIR / f"phase31b_fp_precision_tradeoff_{RUN_OUTPUT_TAG}.png")

print("Figures saved to:", FIGURES_DIR)

Figures saved to: /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/reports/figures


## 11. Metadata and Conclusion

In [12]:
run_metadata = {
    "phase": "3.1b",
    "run_mode": RUN_MODE,
    "sample_rows": SAMPLE_ROWS_LABEL,
    "selected_risk_model": SELECTED_RISK_MODEL,
    "input_paths": {name: str(path.relative_to(PROJECT_ROOT)) for name, path in PATHS.items() if path.exists()},
    "candidate_policies": [
        "level4_amount_bin_dynamic_threshold_v1",
        "level4_shrunk_amount_bin_threshold",
        "level4_precision_guard_threshold",
        "level4_bin_strategy_grid",
    ],
    "selectors": [
        "level4_tuned_best_cost_selector",
        "level4_tuned_guarded_selector",
    ],
    "hyperparameter_grids": {
        "lambda_grid": LAMBDA_GRID,
        "precision_floor_grid": PRECISION_FLOOR_GRID,
        "fp_cost_cap_multiplier_grid": FP_COST_CAP_MULTIPLIER_GRID,
        "threshold_grid_size": int(len(THRESHOLD_GRID)),
        "guarded_selector_tolerance": GUARDED_SELECTOR_TOLERANCE,
    },
    "leakage_controls": {
        "candidate_tuning": "validation_only",
        "test_usage": "report_only_after_candidate_selection",
        "amount_bin_boundaries": "phase31_saved_or_validation_amounts_only",
        "closed_form_dynamic_claim": "forbidden_transactionamt_cancels",
        "diagnostic_modified_cost_model": "not_implemented_in_mvp",
    },
    "outputs": {
        "candidate_metrics": str(CANDIDATE_METRICS_PATH.relative_to(PROJECT_ROOT)),
        "candidate_thresholds": str(CANDIDATE_THRESHOLDS_PATH.relative_to(PROJECT_ROOT)),
        "selected_policies": str(SELECTED_POLICIES_PATH.relative_to(PROJECT_ROOT)),
        "disagreement_cases": str(DISAGREEMENT_PATH.relative_to(PROJECT_ROOT)),
        "five_level_comparison_tuned": str(TUNED_COMPARISON_PATH.relative_to(PROJECT_ROOT)),
    },
}

for path in [
    RESULTS_DIR / "phase31b_run_metadata.json",
    RESULTS_DIR / f"phase31b_run_metadata_{RUN_OUTPUT_TAG}.json",
]:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run_metadata, f, ensure_ascii=False, indent=2)

guarded_test = selected_test_for_comparison[
    selected_test_for_comparison["policy"].eq("level4_tuned_guarded_selector")
].copy()

phase31_level4_test = candidate_metrics_all_df[
    candidate_metrics_all_df["split"].eq("test")
    & candidate_metrics_all_df["policy"].eq("level4_amount_bin_dynamic_threshold_v1")
][["cost_config", "precision_fraud", "fp_cost", "total_cost", "cost_saving_vs_approve_all"]].rename(columns={
    "precision_fraud": "phase31_precision",
    "fp_cost": "phase31_fp_cost",
    "total_cost": "phase31_total_cost",
    "cost_saving_vs_approve_all": "phase31_cost_saving",
})
compare = guarded_test.merge(phase31_level4_test, on="cost_config", how="left")
compare["precision_delta"] = compare["precision_fraud"] - compare["phase31_precision"]
compare["fp_cost_delta"] = compare["fp_cost"] - compare["phase31_fp_cost"]
compare["total_cost_delta"] = compare["total_cost"] - compare["phase31_total_cost"]

print("Guarded selector vs Phase 3.1 Level 4")
display(compare[[
    "cost_config", "selected_candidate_policy", "recall_fraud", "precision_fraud",
    "phase31_precision", "precision_delta", "fp_cost_delta",
    "total_cost_delta", "cost_saving_vs_approve_all"
]].sort_values("cost_config"))

print("Conclusion guard:")
print("- Phase 3.1b tunes Level 4 only; it does not implement Level 5 or retrain LightGBM.")
print("- Candidate choices are made on validation only; test is report-only.")
print("- The original expected-cost threshold is not amount-dynamic because TransactionAmt cancels.")
print("- If tuned selectors improve Cost-A/B precision or FP Cost, report this as reduced over-blocking.")
print("- If Cost-C remains strongest for the original Level 4, report the trade-off honestly.")

Guarded selector vs Phase 3.1 Level 4


,cost_config,selected_candidate_policy,recall_fraud,precision_fraud,phase31_precision,precision_delta,fp_cost_delta,total_cost_delta,cost_saving_vs_approve_all
0,Cost-A,level4_shrunk_amount_bin_threshold,0.645902,0.189060,0.148368,0.040692,-749.288149,-695.070150,17110.100182
1,Cost-B,level4_shrunk_amount_bin_threshold,0.645902,0.189060,0.148368,0.040692,-1498.576298,-1390.140300,34220.200364
2,Cost-C,level4_bin_strategy_grid,0.711475,0.113791,0.111623,0.002168,-2890.231197,-2665.036189,87381.075147


Conclusion guard:
- Phase 3.1b tunes Level 4 only; it does not implement Level 5 or retrain LightGBM.
- Candidate choices are made on validation only; test is report-only.
- The original expected-cost threshold is not amount-dynamic because TransactionAmt cancels.
- If tuned selectors improve Cost-A/B precision or FP Cost, report this as reduced over-blocking.
- If Cost-C remains strongest for the original Level 4, report the trade-off honestly.
